[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/04_generalizzazione_overfitting.ipynb)

# Quando il modello si illude di essere bravo

Un modello può sembrare bravissimo sui dati che ha già visto. Ma il vero obiettivo del machine learning è prevedere bene su casi nuovi. In questo episodio introduciamo training set, test set, generalizzazione e overfitting.

> **Missione** — Separare i dati in allenamento e test, confrontare modelli diversi e capire quando un modello sta imparando una regola oppure sta memorizzando i dati.


## Un sospetto da affrontare

Finora abbiamo misurato l'errore dei nostri modelli sugli **stessi dati** che li avevano addestrati. È come dare a uno studente un compito identico a quello di ripasso, e poi dichiararlo bravissimo perché ha preso dieci. Ma se gli proponessimo un esercizio mai visto?

Apriamo per l'ultima volta il dataset: stavolta non aggiungeremo features, ma cambieremo radicalmente il modo in cui giudichiamo i modelli.


In [ ]:
# Librerie necessarie per scaricare ed estrarre il dataset
import urllib.request, zipfile
from pathlib import Path

# Librerie per l'analisi dei dati e la visualizzazione
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Link al dataset e percorsi locali
DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)
raw_data = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw_data.shape[0]} giocatori, {raw_data.shape[1]} colonne")
raw_data.head()


Stessa pulizia di sempre: il rito d'apertura della serie.


In [ ]:
# Definiamo un sottoinsieme di colonne che ci interessano
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

# Selezioniamo le colonne disponibili e creiamo una copia di lavoro
dataset = raw_data[wanted_columns].copy()

# Manteniamo solo i giocatori con valore noto e positivo
dataset = dataset.dropna(subset=["value_eur", "age", "overall", "potential"])
dataset = dataset[dataset["value_eur"] > 0]

# Rimuoviamo l'1% di giocatori più costosi
dataset = dataset[dataset["value_eur"] <= dataset["value_eur"].quantile(0.99)]

# Per leggibilità: valore in milioni di euro, salario in migliaia
dataset["value_mln_eur"] = dataset["value_eur"] / 1_000_000
if "wage_eur" in dataset.columns:
    dataset["wage_k_eur"] = dataset["wage_eur"] / 1_000

print("Forma del dataset pulito:", dataset.shape)
dataset.head(10)


Strumenti di oggi: la regressione lineare già nota, un nuovo algoritmo molto più flessibile chiamato **albero decisionale** — ne parleremo tra poco — e, soprattutto, la novità concettuale del notebook: `train_test_split`. È la funzione di sklearn che divide a caso i dati in due gruppi, uno per addestrare e uno per giudicare. Senza questa divisione, tutto quello che abbiamo fatto finora era un giudizio in casa.


In [ ]:
# Strumenti di oggi:
# - LinearRegression: il modello che già conosciamo
# - DecisionTreeRegressor: un nuovo algoritmo molto più flessibile
# - le tre metriche di errore
# - train_test_split: la novità fondamentale di questo notebook
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.model_selection import train_test_split


> **Generalizzazione** — Un modello generalizza quando funziona bene su dati nuovi, non solo sui dati usati per addestrarlo. Questo e' il cuore del machine learning: non vogliamo una memoria dei casi passati, vogliamo una regola utile per il futuro.


### Errore di training vs errore atteso

Quello che vorremmo davvero misurare è quanto sbaglierà il modello sui **casi futuri** — i giocatori che il direttore sportivo dovrà valutare la prossima estate. Matematicamente è il cosiddetto **errore atteso**, una media "su tutti i possibili casi del mondo":

$\displaystyle R(f) \;=\; \mathbb{E}_{(x, y)}\!\left[(y - f(x))^2\right]$

Non possiamo calcolarlo esattamente, perché non conosciamo "tutti" i casi del futuro. Quello che possiamo calcolare è l'**errore empirico** sui dati che abbiamo a disposizione:

$\displaystyle \hat{R}_{\text{train}}(f) \;=\; \frac{1}{n_{\text{train}}} \sum_{i \in \text{train}} (y_i - f(x_i))^2$

Il punto delicato è che $\hat{R}_{\text{train}}$ può ingannarci. Un modello molto flessibile può renderlo quasi zero "imparando a memoria" il training set, restando però pessimo sui dati nuovi. Per stimare onestamente $R(f)$ serve misurare l'errore su un **test set** mai visto durante l'addestramento:

$\displaystyle \hat{R}_{\text{test}}(f) \;=\; \frac{1}{n_{\text{test}}} \sum_{i \in \text{test}} (y_i - f(x_i))^2$

È questo numero — non l'errore di training — che dobbiamo guardare per giudicare se il modello generalizza.


## Dividiamo i dati: training e test

Estraiamo a caso il **25%** dei giocatori e li mettiamo da parte: saranno il nostro test set, calciatori che il modello non vedrà mai durante l'addestramento. Il parametro `random_state=42` non ha nulla di magico: serve solo a fissare il seme del generatore casuale, così la divisione sarà identica per tutta la classe e potremo confrontare i risultati.


In [ ]:
# Selezioniamo le feature da usare come input.
# Il list comprehension le prende solo se esistono nel dataset.
features = [
    c for c in ["age", "overall", "potential", "wage_eur"]
    if c in dataset.columns
]
X = dataset[features]
y = dataset["value_mln_eur"]

# train_test_split divide casualmente i dati in due parti:
# - 75% per addestrare (training set)
# - 25% per valutare il modello su dati che non ha mai visto (test set)
# random_state=42 garantisce che la divisione sia riproducibile.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Esempi di training:", len(X_train))
print("Esempi di test:", len(X_test))


> **Training set e test set** — Il training set serve per imparare i parametri del modello. Il test set resta nascosto fino alla valutazione finale. Se il modello va bene anche sul test set, abbiamo piu' fiducia che abbia imparato una regola generale.


## Modello lineare valutato correttamente

Iniziamo dalla regressione lineare che già conosciamo bene. Stavolta però la addestriamo *solo* sul training set, e misuriamo le sue prestazioni separatamente sui due insiemi. Se il modello generalizza, gli errori sui due insiemi dovrebbero essere simili.


In [ ]:
# Addestriamo il modello SOLO sul training set
lin = LinearRegression()
lin.fit(X_train, y_train)

# Calcoliamo le previsioni separatamente sui due insiemi
pred_train_lin = lin.predict(X_train)
pred_test_lin = lin.predict(X_test)

# Funzione di comodo per calcolare le tre metriche tutte insieme
def metrics(y_true, y_pred):
    return {
        "MAE_mln": mean_absolute_error(y_true, y_pred),
        "RMSE_mln": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

# Raccogliamo i risultati in una lista che useremo per confrontare i modelli
results = []
results.append({
    "modello": "Lineare", "dati": "training",
    **metrics(y_train, pred_train_lin),
})
results.append({
    "modello": "Lineare", "dati": "test",
    **metrics(y_test, pred_test_lin),
})
pd.DataFrame(results).round(3)


## Un modello molto flessibile: albero decisionale

Adesso facciamo entrare in scena un *rivale* del modello lineare: l'**albero decisionale**. È un algoritmo molto più flessibile, capace di adattarsi a relazioni complicate fatte di soglie e regole annidate ("se overall > 80 e age < 25 allora valore alto, altrimenti..."). Lo lasciamo crescere libero, senza limiti di profondità: vediamo cosa succede.


In [ ]:
# DecisionTreeRegressor senza limiti: l'albero può crescere quanto vuole
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

# Previsioni separate sui due insiemi
pred_train_tree = tree.predict(X_train)
pred_test_tree = tree.predict(X_test)

# Aggiungiamo i risultati alla nostra tabella di confronto
results.append({
    "modello": "Albero non limitato", "dati": "training",
    **metrics(y_train, pred_train_tree),
})
results.append({
    "modello": "Albero non limitato", "dati": "test",
    **metrics(y_test, pred_test_tree),
})
pd.DataFrame(results).round(3)


> **Overfitting** — Overfitting significa che il modello si adatta troppo ai dettagli del training set. In quel caso l'errore sul training puo' diventare molto basso, ma l'errore sul test resta alto o peggiora. E' come uno studente che impara a memoria gli esercizi svolti ma non sa risolverne uno nuovo.


## Limitiamo la complessità dell'albero

L'albero senza limiti era così bravo sul training quasi quanto il libro di soluzioni — e così meno bravo sul test. Proviamo a *potarlo*: gli imponiamo una **profondità massima** di $4$. Sarà meno flessibile dell'albero senza limiti e quindi probabilmente sbaglierà di più sul training, ma forse memorizzerà meno il rumore e farà meglio sui dati nuovi.


In [ ]:
# Stesso albero, ma con max_depth=4: non potrà avere più di 4 livelli
tree_small = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_small.fit(X_train, y_train)

pred_train_small = tree_small.predict(X_train)
pred_test_small = tree_small.predict(X_test)

results.append({
    "modello": "Albero max_depth=4", "dati": "training",
    **metrics(y_train, pred_train_small),
})
results.append({
    "modello": "Albero max_depth=4", "dati": "test",
    **metrics(y_test, pred_test_small),
})

pd.DataFrame(results).round(3)


> **Complessita' del modello** — Un modello troppo semplice puo' non catturare la struttura dei dati: underfitting. Un modello troppo complesso puo' catturare anche rumore e casi particolari: overfitting. L'obiettivo e' trovare un equilibrio che funzioni bene sui dati nuovi.


## Vediamo l'overfitting come una curva


### La curva a U dell'overfitting

Quanto abbiamo visto finora — albero libero brillante in casa ma debole in trasferta, albero potato un po' meno brillante ma più affidabile — è un caso particolare di un fenomeno generale. Aumentando la complessità di un modello (per esempio la profondità massima dell'albero) succede tipicamente che:

- L'**errore di training** cala sempre: più il modello è flessibile, più si adatta ai dati visti.
- L'**errore di test** prima cala (il modello sta imparando la regola vera), poi a un certo punto **risale** (il modello sta imparando il rumore del training).

Il punto migliore è il **minimo della curva di test**. È il classico equilibrio tra:

- **Bias** (errore sistematico): un modello troppo semplice non riesce a catturare il vero pattern dei dati — è il caso dell'*underfitting*.
- **Varianza**: un modello troppo flessibile cambia molto al variare dei dati di training — è il caso dell'*overfitting*.

Ora vediamolo con i nostri occhi.


Per vedere il fenomeno dal vivo, ripetiamo l'esperimento dell'albero variando la profondità da $1$ a $20$. Per ogni valore registriamo l'errore sul training e quello sul test, poi disegniamo le due curve sullo stesso grafico. Se la teoria dice il vero, dovremmo vedere una curva di test che prima scende e poi risale.


In [ ]:
# Profondità da 1 a 20: una lista di valori da provare
profondita = list(range(1, 21))
err_train = []
err_test = []

# Per ogni profondità: costruiamo l'albero, lo addestriamo, misuriamo
# l'errore sui due insiemi e salviamo i risultati nelle due liste.
for d in profondita:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    err_train.append(mean_absolute_error(y_train, m.predict(X_train)))
    err_test.append(mean_absolute_error(y_test, m.predict(X_test)))

# Disegniamo le due curve sullo stesso grafico per confrontarle
plt.figure(figsize=(8,5))
plt.plot(
    profondita, err_train, "o-",
    label="Errore su training", color="#2a9d8f",
)
plt.plot(
    profondita, err_test, "o-",
    label="Errore su test", color="#e76f51",
)
plt.xlabel("Profondita' massima dell'albero")
plt.ylabel("MAE [milioni di euro]")
plt.title("Underfitting vs overfitting al crescere della complessita'")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Troviamo (e stampiamo) la profondità con l'errore di test minimo:
# è il "sweet spot" tra underfitting e overfitting.
best_d = profondita[int(np.argmin(err_test))]
print(f"Profondita' con errore di test minimo: {best_d}")


## Previsioni finali su casi di test

Confrontiamo le previsioni dei tre modelli — lineare, albero libero, albero limitato — su un campione di giocatori del **test set**. Cioè calciatori che nessuno dei tre modelli ha mai visto durante l'addestramento. È la prova del nove dell'intero notebook: chi è davvero lo scout più affidabile?


In [ ]:
# Costruiamo una vista che mette fianco a fianco le previsioni
# dei tre modelli sui giocatori del test set.
test_view = X_test.copy()
test_view["valore_reale"] = y_test
test_view["lineare"] = pred_test_lin
test_view["albero_non_limitato"] = pred_test_tree
test_view["albero_limitato"] = pred_test_small

# Recuperiamo i nomi dei giocatori corrispondenti dall'indice originale
names = dataset.loc[
    test_view.index,
    ["short_name", "club_name", "player_positions"],
]

# Uniamo nomi e previsioni in un'unica tabella
final_view = pd.concat([names, test_view], axis=1)

# Mostriamo 12 giocatori a caso del test set, arrotondati a 2 decimali
final_view.sample(12, random_state=5).round(2)


---

> **Discussione finale** — Quale modello scegliereste per aiutare davvero una societa' sportiva? Non basta dire quello con errore minore sul training. Bisogna guardare il test set e ragionare sulla stabilita' del modello.


---

> **Cosa abbiamo imparato nella serie** — Abbiamo trasformato una domanda concreta, quanto vale un calciatore?, in un problema di regressione. Abbiamo esplorato dati, costruito un primo modello, aggiunto variabili, misurato errori e infine distinto tra prestazione apparente e generalizzazione.
